# Plant Disease Detection using CNN & Explainability (Grad-CAM)
### Exploratory Data Analysis, Data Augmentation, and Model Experiments

This notebook demonstrates:
1. **Dataset Exploration & Class Distribution**
2. **Preprocessing & Visualizing Data Augmentation** (Rotation, Flip, Zoom)
3. **Model Architectures**: Custom Deep CNN baseline vs. Transfer Learning (MobileNetV2)
4. **Training Dynamics & Loss/Accuracy Curves**
5. **Evaluation Metrics & Confusion Matrix**
6. **Grad-CAM Explainability**: Visualizing lesion attention maps on plant leaves

In [ ]:
import os
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Set project paths
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
print(f"Project Root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Exploratory Data Analysis (EDA)
Let's inspect the class folders in the dataset and examine the split distributions.

In [ ]:
train_dir = DATA_DIR / "train"
classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
counts = [len(list((train_dir / c).glob("*.jpg"))) for c in classes]

plt.figure(figsize=(12, 6))
bars = plt.barh([c.replace("___", " - ") for c in classes], counts, color="#2ca02c")
plt.xlabel("Number of Images", fontsize=11, fontweight="bold")
plt.title("Class Distribution in Training Split", fontsize=13, fontweight="bold")
plt.gca().invert_yaxis()
plt.grid(axis="x", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

## 2. Preprocessing & Data Augmentation
We apply standard ImageNet normalization, along with **Random Rotation (up to 30°)**, **Horizontal and Vertical Flips**, and **Random Resized Crop (Zoom)**.

In [ ]:
from torchvision import transforms
from src.data_loader import get_transforms

train_tf, eval_tf = get_transforms(image_size=224)

# Sample an image
sample_class = classes[0]
sample_img_path = list((train_dir / sample_class).glob("*.jpg"))[0]
orig_img = Image.open(sample_img_path).convert("RGB")

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(orig_img)
axes[0].set_title("Original Leaf")
axes[0].axis("off")

for i in range(1, 5):
    aug_tensor = train_tf(orig_img)
    # Denormalize for display
    from src.data_loader import denormalize_image
    aug_np = denormalize_image(aug_tensor)
    axes[i].imshow(aug_np)
    axes[i].set_title(f"Augmented #{i}")
    axes[i].axis("off")

plt.suptitle("Data Augmentation Pipeline (Rotation, Flips, Zoom, Color Jitter)", fontsize=12, y=1.05)
plt.show()

## 3. Model Architecture Comparison
We compare a **Custom Deep CNN** baseline vs. **Transfer Learning with MobileNetV2**.

In [ ]:
from src.model import build_model

custom_cnn = build_model("custom_cnn", num_classes=len(classes))
mobilenet = build_model("mobilenet_v2", num_classes=len(classes))

n_params_cnn = sum(p.numel() for p in custom_cnn.parameters())
n_params_mob = sum(p.numel() for p in mobilenet.parameters())

print(f"Custom CNN Parameters:      {n_params_cnn:,}")
print(f"MobileNetV2 Parameters:     {n_params_mob:,}")

## 4. Evaluation and Visual Explainability (Grad-CAM)
Grad-CAM (Gradient-weighted Class Activation Mapping) computes the gradients of the score for class $c$ with respect to the feature activation map $A^k$ of the final convolutional layer.

In [ ]:
from src.gradcam import GradCAM, apply_gradcam_overlay

# Run Grad-CAM on a test image
test_input = eval_tf(orig_img).unsqueeze(0)
grad_cam = GradCAM(mobilenet)
heatmap, pred_idx, conf = grad_cam.generate_heatmap(test_input)
grad_cam.remove_hooks()

overlay = apply_gradcam_overlay(orig_img, heatmap, alpha=0.55, colormap_name="jet")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
ax1.imshow(orig_img)
ax1.set_title("Original Leaf")
ax1.axis("off")

ax2.imshow(heatmap, cmap="jet")
ax2.set_title("Grad-CAM Activation Map")
ax2.axis("off")

ax3.imshow(overlay)
ax3.set_title(f"Overlay (Confidence: {conf*100:.1f}%)")
ax3.axis("off")

plt.tight_layout()
plt.show()